## Circle Rasterisation

#### Objective
- $\text{To develop an efficient algorithm which draws a circular curve - Midpoint Drawing Algorithm}$



- $\text{We'll use an implicit representation of a circle to develop the Midpoint Drawing Algorithm}$
- $$ (x - x_0)^2 + (y - y_0)^2 - r^2 = 0 \ | \ (x_0, y_0) \text{ is the centre of the cirle and r is the radius}$$
- $\text{We'll apply a simple translation from the origin and scale from the unit size, in order the represent any circle}$
- $\text{Thus we assume that centre =} (0,0) \text{ with a radius r}$
$$ x^2 + y^2 - r^2 = 0 $$

<p align="center">
    <img src="image_U11/Screenshot 2025-06-20 at 16.18.17.png" height="400" width="500"/>

</p>

1. $\text{We start at the pixel (-r, 0) and iterate the x coordinate to the right by increments of 1 (-r, -r +1, -r +2, ....)}$
$\text{For a given x we search for a point that lies on the circle and since it's a closed curve we have two pixel for every x-value}$
2. $ y = \pm \sqrt{r^2 - x^2}$
### Psuedo-Code
```cpp
r2 = r*r;
for x = -r to r {
    y = sqrt(r2 - x^2);
    SetPixel(x, Round(y));
    SetPixel(x, -Round(y));
}
```

#### Issues 
1. $\text{We're using expensive operations (float points, and round)}$
2. $\text{The gradient of the circle changes as x changes increase and decreases which as mentioned earlier can cause gaps in the rasterisation}$

<p align="center">
    <img src="image_U11/Screenshot 2025-06-20 at 16.29.01.png" height="400" width="500"/>
    <img src="image_U11/Screenshot 2025-06-20 at 16.29.11.png" height="400" width="400"/>
<p/>

- $\text{A potential solution is to alternate the step/iteration depending on the gradient of the curve}$
$\text{If the gradient is low we'll step in the x-direction and if the gradient is high we'll step in the y-direction}$

- $\text{A simpler solution we'll use the symmetric properties of a circle}$
$\text{ We know for a coordinate (x,y) ON the circle exists coordinate (-x, y), (x, -y) and (-x, -y)}$
$\text{We can imply further points if (x,y) is ON the circle then (y,x) wich implies (-y, x), (y, -x) and (-y, -x)}$
$\text{This means we only need the algorithm to compute an eighth of a circle}$
1. $\text{Suppose we started on (0, r) and iterate over the x axis (since the gradient is low) until we reach the 45 degree where m=1}$
2. $\text{Using the 7 more points we'll be able to complete the circle}$

<p align="center">
    <img src="image_U11/Screenshot 2025-06-20 at 16.31.09.png" height="400" width="400"/>
    <img src="image_U11/Screenshot 2025-06-20 at 16.41.44.png" height="400" width="400"/>
    <img src="image_U11/Screenshot 2025-06-20 at 16.41.58.png" height="400" width="400"/>
<p/>

---



- $\text{Given that we're starting in the (0, r) position we have two options:}$ 

 $\text{1. Either to move in the x+ direction (east)}$

 $\text{2. Move down by 1 and east direction (south-east)}$

- $\text{In each step we decide to move east or souteast depending on the pixel that's closer to the cicle }$

<p align="center">
    <img src="image_U11/Screenshot 2025-06-20 at 16.44.27.png" height="400" width="500"/>
    <img src="image_U11/Screenshot 2025-06-20 at 16.44.36.png" height="400" width="500"/>
<p/>

- $\text{To determine the which pixel is closer we could:}$
##### Naive Approach

$$ y = Round(\sqrt(r^2 - x^2))$$

- $\text{The issue with this is that we're using all the computations that lead to complex running time in the first place - defeating the point}$


##### Optimised Approach 

$\text{for x+1 we can rasterise pixel y+1 or pixel y}$

$\text{We create determine the (continous) midpoint between two y-values} \frac{y+1-y}{2} \\ \text{ and using the implicit form of the circle }$ $$ f(x,y) = x^2 + y^2 - r^2$$  $\text{we determine whether the midpoint is inside or outside the circle}$

$\text{if the midpint (f(x,y) > 0) is outside the circle - CHOOSE y}$

$\text{If the midpoint (f(x,y) ≤ 0) is inside the circle - CHOOSE y+1}$

<p align="center">
    <img src="image_U11/Screenshot 2025-06-20 at 16.45.33.png" height="400" width="500"/>
    <img src="image_U11/Screenshot 2025-06-20 at 16.46.57.png" height="400" width="500"/>
<p/>

$\text{Using forward differencing, all we need to do is add the difference between the current function value and the next function value}$

$\text{current pixel =} (x_i, y_i) \text{ and current midpoint function value at } (x_{i}+1, y_i-0.5) = d$
$\text{if d < 0, we must move east and update d by the difference between the function value at the current midpoint and the next midpoint}$ 
$$NewMid Point = (x_{i}+1, y_i-0.5) + (x_{i}+2, y_i-0.5) \rightarrow \Delta_{E} = 2x_i +3$$
$\text{if d > 0, we must move Sout-East, and update d by the difference between the function value at current midpoint and the next midpoint}$
$$NewMid Point = (x_{i}+1, y_i-0.5) + (x_{i}+2, y_i-1.5) \rightarrow \Delta_{SE} = 2(x_i-y_i) +5$$

<p align="center">
    <img src="image_U11/Screenshot 2025-06-20 at 16.49.13.png" height="400" width="500"/>
    <img src="image_U11/Screenshot 2025-06-20 at 16.56.13.png" height="400" width="500"/>
<p/>

- $\text{If we start at (0,r)}$
$$ d = f(0+1, r-0.5) =  \frac{5}{4} - r $$
- $\text{however, to convert the algorthm to work with integers we initialise with }$ $$d = 1 - r$$

``` cpp
int x := 0; 
int y := r;
int d := 1-r;
Plot8SymmetryPixel(x,y);
while (y > x) do {
    if (0 < d ) then {/* East */
        d := d + 2x + 3;
        x := x+1; 
    }
    else { /* South */ 
        d := d + 2(x-y) + 5;
        x := x+1;
        y := y-1;    
    }
    Plot8SymmetryPixel(x,y);
}
```